In [2]:
from tokenizers import Tokenizer
from datasets import load_dataset
import math
import os
from typing import Tuple

import torch
from torch import nn, Tensor
from torch.nn import TransformerEncoder, TransformerEncoderLayer
from torch.utils.data import dataset

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

## Load code from part 2 needed to evaluate

In [3]:
tokenizers = {
    "char":      "trained_tokenizers/char_tokenizer.json",
    "bpe_small": "trained_tokenizers/small_bpe_tokenizer.json",
    "bpe_large": "trained_tokenizers/large_bpe_tokenizer.json",
}

In [21]:
class TransformerModel(nn.Module):

    def __init__(self, ntoken: int, d_model: int, nhead: int, d_hid: int,
                 nlayers: int, dropout: float = 0.5):
        super().__init__()
        self.model_type = 'Transformer'
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        encoder_layers = TransformerEncoderLayer(d_model, nhead, d_hid, dropout)
        self.transformer_encoder = TransformerEncoder(encoder_layers, nlayers)
        self.embedding = nn.Embedding(ntoken, d_model)
        self.d_model = d_model
        self.linear = nn.Linear(d_model, ntoken)

        self.init_weights()

    def init_weights(self) -> None:
        initrange = 0.1
        self.embedding.weight.data.uniform_(-initrange, initrange)
        self.linear.bias.data.zero_()
        self.linear.weight.data.uniform_(-initrange, initrange)

    def forward(self, src: Tensor, src_mask: Tensor = None) -> Tensor:
        """
        Arguments:
            src: Tensor, shape ``[seq_len, batch_size]``
            src_mask: Tensor, shape ``[seq_len, seq_len]``

        Returns:
            output Tensor of shape ``[seq_len, batch_size, ntoken]``
        """
        src = self.embedding(src) * math.sqrt(self.d_model)
        src = self.pos_encoder(src)
        if src_mask is None:
            """Generate a square causal mask for the sequence. The masked positions are filled with float('-inf').
            Unmasked positions are filled with float(0.0).
            """
            src_mask = nn.Transformer.generate_square_subsequent_mask(len(src)).to(device)
        output = self.transformer_encoder(src, src_mask)
        output = self.linear(output)
        return output

class PositionalEncoding(nn.Module):

    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x: Tensor) -> Tensor:
        """
        Arguments:
            x: Tensor, shape ``[seq_len, batch_size, embedding_dim]``
        """
        x = x + self.pe[:x.size(0)]
        return self.dropout(x)


def batchify(data: Tensor, bsz: int) -> Tensor:
    """Divides the data into ``bsz`` separate sequences, removing extra elements
    that wouldn't cleanly fit.

    Arguments:
        data: Tensor, shape ``[N]``
        bsz: int, batch size

    Returns:
        Tensor of shape ``[N // bsz, bsz]``
    """
    seq_len = data.size(0) // bsz
    data = data[:seq_len * bsz]
    data = data.view(bsz, seq_len).t().contiguous()
    return data.to(device)


def data_process(split, lang, tok):
    ids = []
    num_chars = 0
    with open(f"/srv/data/lt2326-h26/a1/{split}/{lang}.txt", encoding="utf-8") as f:
        lines = [l.strip() for l in f if l.strip()]
        num_chars += sum(len(l) for l in lines)
    for e in tok.encode_batch(lines):
        ids.extend(e.ids)
    
    return torch.tensor(ids, dtype=torch.long),num_chars


In [5]:
bptt = 128

def get_batch(source: Tensor, i: int) -> Tuple[Tensor, Tensor]:
    """
    Args:
        source: Tensor, shape ``[full_seq_len, batch_size]``
        i: int

    Returns:
        tuple (data, target), where data has shape ``[seq_len, batch_size]`` and
        target has shape ``[seq_len * batch_size]``
    """
    seq_len = min(bptt, len(source) - 1 - i)
    data = source[i:i+seq_len]
    target = source[i+1:i+1+seq_len].reshape(-1)
    return data, target

In [6]:
def evaluate(model: nn.Module, eval_data: Tensor) -> float:
    model.eval()  # turn on evaluation mode
    total_loss = 0.
    with torch.no_grad():
        for i in range(0, eval_data.size(0) - 1, bptt):
            data, targets = get_batch(eval_data, i)
            seq_len = data.size(0)
            output = model(data)
            output_flat = output.view(-1, ntokens)
            total_loss += seq_len * criterion(output_flat, targets).item()
    return total_loss / (len(eval_data) - 1) # returns average nats per token

In [12]:
criterion = nn.CrossEntropyLoss()
emsize = 256  # embedding dimension
d_hid = 1024  # dimension of the feedforward network model in ``nn.TransformerEncoder``
nlayers = 2  # number of ``nn.TransformerEncoderLayer`` in ``nn.TransformerEncoder``
nhead = 4  # number of heads in ``nn.MultiheadAttention``
dropout = 0.2  # dropout probability
eval_batch_size = 10

## Evaluate

In [28]:
for tokenizer_name,path in tokenizers.items():
    all_nats = 0
    all_chars = 0
    print("Tokenizer: ", tokenizer_name)
    tokenizer_from_file = Tokenizer.from_file(path)
    ntokens = tokenizer_from_file.get_vocab_size()
    model = TransformerModel(ntokens, emsize, nhead, d_hid, nlayers, dropout).to(device)
    model.load_state_dict(torch.load(f"models/{tokenizer_name}_best.pt"))
    model.eval()

    for lang in ["en", "tr", "zh"]:
        print("Language: ", lang)
        test_data,num_chars = data_process("test", lang, tokenizer_from_file)
        all_chars += num_chars
        
        test_data_batch = batchify(test_data, eval_batch_size)
    
        total_nats = evaluate(model, test_data_batch) * (len(test_data_batch) - 1) * eval_batch_size
        all_nats += total_nats
        
        bpc = total_nats / (math.log(2) * num_chars)
        print("BPC: ", bpc)

    print("Overall BPC: ", all_nats / (math.log(2) * all_chars))
    print("--------------------------------------------------")

Tokenizer:  char
Language:  en


/tmp/ipykernel_2287783/3570648520.py:9: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer_encoder = TransformerEncoder(encoder_layers, nlayers)


BPC:  2.983023556486434
Language:  tr
BPC:  2.994630528787764
Language:  zh
BPC:  6.905701523321692
Overall BPC:  4.2943090631204255
--------------------------------------------------
Tokenizer:  bpe_small
Language:  en
BPC:  2.233810526882721
Language:  tr
BPC:  2.2704544428454407
Language:  zh
BPC:  6.5886987352322075
Overall BPC:  3.697496815561104
--------------------------------------------------
Tokenizer:  bpe_large
Language:  en
BPC:  2.066602076453999
Language:  tr
BPC:  2.1075152312655647
Language:  zh
BPC:  6.374066419840653
Overall BPC:  3.5159053625742906
--------------------------------------------------


In [ ]:


test_loss = evaluate(model, test_data)
test_ppl = math.exp(test_loss)
print('=' * 89)
print(f'| End of training | test loss {test_loss:5.2f} | '
      f'test ppl {test_ppl:8.2f}')
print('=' * 89)
